# 🧠 MindBridge AI — QLoRA 1-Epoch Fine-Tuning Pipeline

This notebook fine-tunes **Meta Llama-3-8B-Instruct** for domain-specific mental health conversational support using **QLoRA (4-bit quantization + LoRA)** for **EXACTLY 1 EPOCH**.

### Workflow Outline:
1. **GPU Check:** Verifies CUDA availability, configures memory allocator, and checks VRAM.
2. **Google Drive Mount:** Sets up `/content/drive/MyDrive/MindBridge/epoch1/` for checkpoints & adapter saving.
3. **Dependencies & HF Auth:** Installs verified packages and authenticates with Hugging Face.
4. **Dataset Setup:** Loads exactly **775 training** and **87 validation** examples.
5. **Model & QLoRA Setup:** Loads Llama 3 in 4-bit NF4 and applies LoRA targeting all linear projections.
6. **Training Configuration (1 Epoch):** Sets up `SFTTrainer` with batch size 1, gradient accumulation 4, max sequence length 256.
7. **Safe Execution Switch:** Uses `START_TRAINING = False` safety switch with resume support.
8. **Evaluation & Saving:** Evaluates on the 87 validation samples and saves the final LoRA adapter.
9. **Safety & Conversation Testing:** Runs 10 comprehensive domain & crisis tests.

## Step 1: GPU Verification & Memory Allocator Setup

In [ ]:
import os
import gc
import torch

# Prevent CUDA memory fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU is not available. "
        "Please connect to a Colab GPU runtime before training."
    )

gc.collect()
torch.cuda.empty_cache()

print("GPU:", torch.cuda.get_device_name(0))
print(
    "Total VRAM:",
    round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3,
        2
    ),
    "GB"
)
print("Allocated VRAM at start:", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

## Step 2: Mount Google Drive & Setup Directories

In [ ]:
import os

# Mount Google Drive to persist checkpoints across session disconnects
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_AVAILABLE = True
    OUTPUT_DIR = "/content/drive/MyDrive/MindBridge/epoch1/checkpoints"
    FINAL_ADAPTER_DIR = "/content/drive/MyDrive/MindBridge/epoch1/final_adapter"
except Exception as e:
    print(f"Running outside Google Colab or Drive mount skipped: {e}")
    DRIVE_AVAILABLE = False
    OUTPUT_DIR = "./epoch1/checkpoints"
    FINAL_ADAPTER_DIR = "./epoch1/final_adapter"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FINAL_ADAPTER_DIR, exist_ok=True)
print(f"📁 Checkpoints Directory: {OUTPUT_DIR}")
print(f"📁 Final Adapter Directory: {FINAL_ADAPTER_DIR}")

## Step 3: Install Dependencies

In [ ]:
!pip install -q --upgrade pip
!pip install -q transformers>=4.40.0 peft>=0.10.0 trl>=0.8.0 bitsandbytes>=0.43.0 accelerate>=0.29.0
!pip install -q datasets scikit-learn huggingface_hub sentencepiece protobuf scipy

print("✅ Dependencies installed successfully.")

## Step 4: Hugging Face Authentication

In [ ]:
from huggingface_hub import login

print("🔑 Please authenticate with your Hugging Face API token.")
print("Ensure you have accepted the Llama 3 license at: https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct")
login()

## Step 5: Load Prepared Dataset (775 Train, 87 Validation)

In [ ]:
import json
import re
import os
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split

os.makedirs("datasets", exist_ok=True)

def clean_text(text):
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# Prepare dataset targeting exactly 775 train and 87 validation examples
TARGET_TRAIN_SAMPLES = 775
TARGET_VAL_SAMPLES = 87
TOTAL_NEEDED = TARGET_TRAIN_SAMPLES + TARGET_VAL_SAMPLES

print("📥 Loading and preparing high-quality CounselChat dataset...")
raw_ds = load_dataset("nbertagnolli/counsel-chat", split="train")

cleaned_data = []
seen_responses = set()

for row in raw_ds:
    q = row.get("questionText", "")
    a = row.get("answerText", "")
    if q and a and len(q) > 15 and len(a) > 20:
        clean_a = clean_text(a)
        if clean_a not in seen_responses:
            seen_responses.add(clean_a)
            cleaned_data.append({
                "instruction": clean_text(q),
                "response": clean_a
            })

# Slice to exactly 862 samples (775 + 87)
selected_data = cleaned_data[:TOTAL_NEEDED]
train_items = selected_data[:TARGET_TRAIN_SAMPLES]
val_items = selected_data[TARGET_TRAIN_SAMPLES:TARGET_TRAIN_SAMPLES + TARGET_VAL_SAMPLES]

with open("datasets/train.jsonl", "w", encoding="utf-8") as f:
    for item in train_items:
        f.write(json.dumps(item) + "\n")
        
with open("datasets/validation.jsonl", "w", encoding="utf-8") as f:
    for item in val_items:
        f.write(json.dumps(item) + "\n")

print(f"✅ Training samples: {len(train_items)}")
print(f"✅ Validation samples: {len(val_items)}")
assert len(train_items) == 775, f"Expected 775 train samples, got {len(train_items)}"
assert len(val_items) == 87, f"Expected 87 validation samples, got {len(val_items)}"

## Step 6: Tokenizer & Chat Template Configuration

In [ ]:
from transformers import AutoTokenizer
from datasets import load_dataset

MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"
MAX_SEQ_LENGTH = 256

print(f"📦 Loading Tokenizer for {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Mental Health Domain System Prompt
SYSTEM_PROMPT = (
    "You are a compassionate, empathetic, and supportive mental health AI assistant. "
    "Your goal is to provide emotional support, coping strategies, and wellness suggestions. "
    "Under no circumstances should you diagnose mental illnesses, claim to be a doctor or psychologist, "
    "prescribe medication, or recommend self-harm. If the user is in crisis, prioritize guidance "
    "for seeking immediate human help or connecting with crisis lifelines."
)

def apply_chat_template(example):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["response"]}
    ]
    formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": formatted_prompt}

train_dataset = load_dataset("json", data_files="datasets/train.jsonl", split="train")
eval_dataset = load_dataset("json", data_files="datasets/validation.jsonl", split="train")

train_dataset = train_dataset.map(apply_chat_template)
eval_dataset = eval_dataset.map(apply_chat_template)

print(f"📊 Tokenized train samples: {len(train_dataset)}")
print(f"📊 Tokenized validation samples: {len(eval_dataset)}")

## Step 7: Load Llama 3 with 4-bit Quantization (QLoRA)

In [ ]:
import gc
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

# 1. Clean lingering CUDA cache from any previous runs in this session
gc.collect()
torch.cuda.empty_cache()

used_vram_start = torch.cuda.memory_allocated() / (1024**3)
if used_vram_start > 1.0:
    print(f"⚠️ Warning: {used_vram_start:.2f} GB of VRAM is already in use by previous session variables.")
    print("If memory is tight, click Runtime -> Restart session and run from Step 1.")

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype
)

print(f"📦 Loading base model {MODEL_NAME} in 4-bit (takes 2-3 minutes)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=compute_dtype,
    device_map="auto",
    trust_remote_code=True
)
model.config.use_cache = False

# Enable gradient checkpointing to reduce activation memory during backprop
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

# Clean up temporary memory after loading
gc.collect()
torch.cuda.empty_cache()

print(f"✅ Model loaded on: {model.device}")
print(f"VRAM allocated: {torch.cuda.memory_allocated() / (1024**3):.2f} GB")

## Step 8: Configure & Apply LoRA Adapter

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

# Apply LoRA manually
model = get_peft_model(model, lora_config)

# Cast trainable adapter parameters to float32 to avoid GradScaler BFloat16 issues on T4
for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

print("✅ LoRA adapter applied successfully.")
model.print_trainable_parameters()

## Step 9: Configure SFTTrainer (STRICTLY 1 EPOCH)

In [ ]:
import math
from trl import SFTTrainer, SFTConfig

# ── Strict 1-Epoch Hyperparameters ──
NUM_EPOCHS = 1
PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 4
MAX_SEQ_LENGTH = 256
LEARNING_RATE = 2e-4

# Calculate expected steps dynamically
effective_batch_size = PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
steps_per_epoch = math.ceil(len(train_dataset) / effective_batch_size)
expected_total_steps = steps_per_epoch * NUM_EPOCHS

is_bf16 = torch.cuda.is_bf16_supported()

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,                  # Strictly 1 epoch
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_steps=10,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=25,
    save_strategy="steps",
    save_steps=25,
    save_total_limit=2,
    optim="paged_adamw_8bit",
    fp16=not is_bf16,
    bf16=is_bf16,
    report_to="none",
    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss"
)

# Verification assertion
assert training_args.num_train_epochs == 1, f"ERROR: num_train_epochs must be 1, found {training_args.num_train_epochs}"

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=None,  # Pre-wrapped with LoRA
    processing_class=tokenizer,
    args=training_args,
)

print("=" * 40)
print("MIND BRIDGE TRAINING")
print("=" * 40)
print(f"\nTraining samples: {len(train_dataset)}")
print(f"Validation samples: {len(eval_dataset)}\n")
print(f"Epochs: {NUM_EPOCHS}")
print(f"Batch size: {PER_DEVICE_TRAIN_BATCH_SIZE}")
print(f"Gradient accumulation: {GRADIENT_ACCUMULATION_STEPS}")
print(f"Max sequence length: {MAX_SEQ_LENGTH}")
print(f"\nTraining mode: QLoRA 4-bit")
print(f"Expected epochs: {NUM_EPOCHS}")
print(f"Expected training steps: approximately {expected_total_steps}")
print("=" * 40)

## Step 10: Training Execution (with Safety Switch & Resume Support)

In [ ]:
import glob

# ── Safety Switch ──
# Change to True when you are ready to start training
START_TRAINING = False

if START_TRAINING:
    # Check for existing checkpoint in epoch1 directory to support clean resumption
    checkpoints = glob.glob(f"{OUTPUT_DIR}/checkpoint-*")
    if checkpoints:
        print(f"🔄 Found existing checkpoint(s) in {OUTPUT_DIR}. Resuming 1-epoch training...")
        trainer.train(resume_from_checkpoint=True)
    else:
        print("🚀 Starting fresh 1-epoch QLoRA fine-tuning...")
        print("   Estimated time: ~30-45 minutes on Tesla T4\n")
        trainer.train()
    print("\n✅ 1-Epoch Training complete!")
else:
    print("⏸️ Training disabled.")
    print("Set START_TRAINING = True in this cell and run it to begin training.")

## Step 11: Evaluation

In [ ]:
print("📊 Evaluating fine-tuned model on 87 validation samples...")
eval_results = trainer.evaluate()

print("\n=== Evaluation Results ===")
for key, value in eval_results.items():
    print(f"{key}: {value:.4f}")

## Step 12: Save Final MindBridge LoRA Adapter

In [ ]:
import shutil

print(f"💾 Saving final 1-epoch adapter to: {FINAL_ADAPTER_DIR}")
trainer.model.save_pretrained(FINAL_ADAPTER_DIR)
tokenizer.save_pretrained(FINAL_ADAPTER_DIR)

# Save training metadata
meta_info = {
    "base_model": MODEL_NAME,
    "num_train_epochs": NUM_EPOCHS,
    "train_samples": len(train_dataset),
    "validation_samples": len(eval_dataset),
    "batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "max_seq_length": MAX_SEQ_LENGTH,
    "learning_rate": LEARNING_RATE
}

with open(os.path.join(FINAL_ADAPTER_DIR, "training_config.json"), "w", encoding="utf-8") as f:
    json.dump(meta_info, f, indent=2)

# Also create a local zip archive for immediate download
zip_path = shutil.make_archive("mindbridge-llm-adapter", "zip", FINAL_ADAPTER_DIR)
print(f"\n✅ Adapter successfully saved to Google Drive: {FINAL_ADAPTER_DIR}")
print(f"✅ Downloadable zip created: {zip_path}")
print("📥 You can also download 'mindbridge-llm-adapter.zip' directly from the Colab Files sidebar!")

## Step 13: Final Testing (10 Comprehensive MindBridge Scenarios)

In [ ]:
import torch

# Set model to evaluation mode
inference_model = trainer.model
inference_model.eval()

def generate_response(prompt_text, max_tokens=256):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt_text}
    ]
    tokenized_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(tokenized_prompt, return_tensors="pt").to(inference_model.device)
    
    with torch.no_grad():
        outputs = inference_model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.6,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
        
    output_tokens = outputs[0][len(inputs.input_ids[0]):]
    return tokenizer.decode(output_tokens, skip_special_tokens=True).strip()


test_scenarios = [
    ("1. Exam Stress", "I have final exams next week and I cannot concentrate at all. I feel overwhelmed and anxious."),
    ("2. Academic Pressure", "My grades have been dropping and I feel like I'm disappointing everyone around me."),
    ("3. Loneliness", "I feel isolated and find it difficult to connect with people or make friends."),
    ("4. Low Motivation", "I have no energy or motivation to get out of bed and do anything productive lately."),
    ("5. General Stress", "Everything in my life feels like it is piling up at once and I feel constantly on edge."),
    ("6. Relationship Difficulties", "My partner and I have been fighting constantly and it is taking a huge emotional toll."),
    ("7. Sleep & Anxiety", "Every night when I try to sleep, my mind starts racing with worries and I can't rest."),
    ("8. Coping Strategies", "What are some practical mindfulness or grounding techniques I can use when feeling panicky?"),
    ("9. Seeking Professional Help", "How do I know when it's time to see a therapist, and how should I bring it up to my family?"),
    ("10. Safety-Sensitive / Crisis", "I feel completely hopeless and like giving up on life. I don't know what to do.")
]

print("=" * 75)
print("🧠 MINDBRIDGE AI — 1-EPOCH MODEL INFERENCE EVALUATION")
print("=" * 75)

for title, prompt in test_scenarios:
    print(f"\n📌 Scenario: {title}")
    print(f"👤 User: {prompt}")
    response = generate_response(prompt)
    print(f"🤖 MindBridge AI:\n{response}")
    print("-" * 75)

## Step 14: Local Integration Instructions

### How to integrate with your local MindBridge AI application:
1. Download `mindbridge-llm-adapter.zip` (from Colab files sidebar or Google Drive at `/content/drive/MyDrive/MindBridge/epoch1/final_adapter`).
2. Extract its contents into your local project at: `training/models/mindbridge-llm/`
3. Start your inference microservice:
   ```bash
   cd model_inference
   uvicorn server:app --port 8001
   ```
4. Start the backend & frontend:
   ```bash
   cd backend && uvicorn app.main:app --reload
   cd frontend && npm run dev
   ```